# EE348 - Knowledge Representation and Reasoning in AI
## Lab 5: Logical Inference and Automated Reasoning in Python

This notebook completes Tasks 1-8 from the lab manual. It implements the reasoning mechanisms directly, prints intermediate inference steps, and explains the logical meaning of each result.

## Task 1 - Rules of Inference in Python

The functions below encode Modus Ponens, Modus Tollens, And-Elimination, And-Introduction, Or-Introduction, and Double Negation using Boolean values.

In [1]:
def modus_ponens(P, P_implies_Q):
    return True if P and P_implies_Q else None

def modus_tollens(not_Q, P_implies_Q):
    return True if not_Q and P_implies_Q else None  # True means not-P follows

def and_elimination(P, Q):
    return P, Q

def and_introduction(P, Q):
    return P and Q

def or_introduction(P, Q=False):
    return P or Q

def double_negation(P):
    return not (not P)

# Lecture example
Rain = True
Rain_implies_Wet = True
Wet_implies_Cancelled = True
Wet = modus_ponens(Rain, Rain_implies_Wet)
Cancelled = modus_ponens(Wet, Wet_implies_Cancelled)
print("Lecture example:")
print("Wet:", Wet)
print("Cancelled:", Cancelled)

# Exercise 1.1: P -> Q, Q -> R, P |- R
P = True
P_implies_Q = True
Q_implies_R = True
Q = modus_ponens(P, P_implies_Q)
R = modus_ponens(Q, Q_implies_R)
print("\nExercise 1.1 trace:")
print("1. P is", P)
print("2. From P and P -> Q, derive Q:", Q)
print("3. From Q and Q -> R, derive R:", R)

# Exercise 1.2: If the alarm is armed, its indicator is on.
# The indicator is not on; therefore, the alarm is not armed.
not_indicator_on = True
armed_implies_indicator_on = True
not_armed = modus_tollens(not_indicator_on, armed_implies_indicator_on)
print("\nExercise 1.2 Modus Tollens:")
print("Indicator is off:", not_indicator_on)
print("Armed -> indicator on:", armed_implies_indicator_on)
print("Therefore, alarm is not armed:", not_armed)

print("\nOther rule checks:")
print("And-Elimination of True AND False:", and_elimination(True, False))
print("And-Introduction:", and_introduction(True, True))
print("Or-Introduction:", or_introduction(True))
print("Double Negation:", double_negation(True))

Lecture example:
Wet: True
Cancelled: True

Exercise 1.1 trace:
1. P is True
2. From P and P -> Q, derive Q: True
3. From Q and Q -> R, derive R: True

Exercise 1.2 Modus Tollens:
Indicator is off: True
Armed -> indicator on: True
Therefore, alarm is not armed: True

Other rule checks:
And-Elimination of True AND False: (True, False)
And-Introduction: True
Or-Introduction: True
Double Negation: True


**Logical meaning.** The lecture facts entail that the match is cancelled. Exercise 1.1 uses two successive applications of Modus Ponens to derive (R). In Exercise 1.2, the false consequent lets us infer the negation of the antecedent by Modus Tollens.

## Task 2 - Literals, Clauses, and CNF

### Part A - Manual CNF conversion

1. (P \rightarrow (Q \land R))
   
   (\equiv \neg P \lor (Q \land R)) - eliminate implication  
   (\equiv (\neg P \lor Q) \land (\neg P \lor R)) - distribute OR over AND  
   **CNF:** ((\neg P \lor Q) \land (\neg P \lor R))

2. (\neg(P \lor Q))
   
   (\equiv \neg P \land \neg Q) - De Morgan's law  
   **CNF:** ((\neg P) \land (\neg Q))

3. ((P \rightarrow Q) \land (Q \rightarrow R))
   
   (\equiv (\neg P \lor Q) \land (\neg Q \lor R)) - eliminate both implications  
   **CNF:** ((\neg P \lor Q) \land (\neg Q \lor R))

4. (P \lor (Q \land R))
   
   (\equiv (P \lor Q) \land (P \lor R)) - distribute OR over AND  
   **CNF:** ((P \lor Q) \land (P \lor R))

In [2]:
cnf_formulas = {
    "1. P -> (Q AND R)": [{"~P", "Q"}, {"~P", "R"}],
    "2. NOT(P OR Q)": [{"~P"}, {"~Q"}],
    "3. (P -> Q) AND (Q -> R)": [{"~P", "Q"}, {"~Q", "R"}],
    "4. P OR (Q AND R)": [{"P", "Q"}, {"P", "R"}],
}

for formula, clauses in cnf_formulas.items():
    print(formula)
    print("  CNF clauses:", clauses)

1. P -> (Q AND R)
  CNF clauses: [{'Q', '~P'}, {'R', '~P'}]
2. NOT(P OR Q)
  CNF clauses: [{'~P'}, {'~Q'}]
3. (P -> Q) AND (Q -> R)
  CNF clauses: [{'Q', '~P'}, {'~Q', 'R'}]
4. P OR (Q AND R)
  CNF clauses: [{'Q', 'P'}, {'P', 'R'}]


**Logical meaning.** Each inner set is a disjunctive clause, while the outer list represents their conjunction. The clause representation preserves each formula's final CNF.

## Task 3 - Propositional Resolution

In [3]:
def complement(literal):
    return literal[1:] if literal.startswith("~") else "~" + literal

def resolve(c1, c2):
    resolvents = []
    for lit in c1:
        comp = complement(lit)
        if comp in c2:
            new_clause = (c1 - {lit}) | (c2 - {comp})
            # Ignore tautological resolvents such as {P, ~P}.
            if not any(complement(x) in new_clause for x in new_clause):
                resolvents.append(new_clause)
    return resolvents

print("Example resolvent:", resolve({"P", "Q"}, {"~P", "R"}))

def resolution_refutation(kb, negated_goal):
    clauses = [frozenset(c) for c in kb] + [frozenset(negated_goal)]
    steps = []
    while True:
        new = set()
        for i in range(len(clauses)):
            for j in range(i + 1, len(clauses)):
                for resolvent in resolve(set(clauses[i]), set(clauses[j])):
                    r = frozenset(resolvent)
                    if r not in clauses and r not in new:
                        pivot = next(
                            (lit for lit in clauses[i] if complement(lit) in clauses[j]),
                            "complementary literals",
                        )
                        steps.append((set(clauses[i]), set(clauses[j]), pivot, set(r)))
                        print(f"Resolve {set(clauses[i])} with {set(clauses[j])} on {pivot} -> {set(r)}")
                        if not r:
                            return True, steps
                        new.add(r)
        if not new:
            return False, steps
        clauses.extend(sorted(new, key=lambda c: (len(c), sorted(c))))

print("\nExercise 3.1: prove R")
# P -> Q is {~P,Q}; Q -> R is {~Q,R}; P is {P}; negate goal R as {~R}.
proved_R, steps_R = resolution_refutation(
    [{"~P", "Q"}, {"~Q", "R"}, {"P"}], {"~R"}
)
print("Empty clause derived; therefore R is proved:", proved_R)

print("\nExercise 3.2: prove C")
proved_C, steps_C = resolution_refutation(
    [{"~A", "B"}, {"~B", "C"}, {"A"}], {"~C"}
)
print("Empty clause derived; therefore C is proved:", proved_C)

Example resolvent: [{'Q', 'R'}]

Exercise 3.1: prove R
Resolve {'Q', '~P'} with {'~Q', 'R'} on Q -> {'R', '~P'}
Resolve {'Q', '~P'} with {'P'} on ~P -> {'Q'}
Resolve {'~Q', 'R'} with {'~R'} on R -> {'~Q'}
Resolve {'Q', '~P'} with {'~Q'} on Q -> {'~P'}
Resolve {'~Q', 'R'} with {'Q'} on ~Q -> {'R'}
Resolve {'Q'} with {'~Q'} on Q -> set()
Empty clause derived; therefore R is proved: True

Exercise 3.2: prove C
Resolve {'B', '~A'} with {'C', '~B'} on B -> {'C', '~A'}
Resolve {'B', '~A'} with {'A'} on ~A -> {'B'}
Resolve {'C', '~B'} with {'~C'} on C -> {'~B'}
Resolve {'B', '~A'} with {'~B'} on B -> {'~A'}
Resolve {'C', '~B'} with {'B'} on ~B -> {'C'}
Resolve {'B'} with {'~B'} on B -> set()
Empty clause derived; therefore C is proved: True


**Logical meaning.** Adding the negation of each goal makes the clause set inconsistent: resolution derives the empty clause `set()`. By proof by refutation, the original knowledge base entails (R) in Exercise 3.1 and (C) in Exercise 3.2.

## Task 4 - Substitution and Unification

In [4]:
VARIABLES = {"x", "y", "z", "w"}

def is_variable(term):
    return isinstance(term, str) and term in VARIABLES

def dereference(term, theta):
    while is_variable(term) and term in theta and theta[term] != term:
        term = theta[term]
    return term

def apply_substitution(expr, theta):
    return tuple(dereference(item, theta) for item in expr)

def unify(expr1, expr2):
    if len(expr1) != len(expr2) or expr1[0] != expr2[0]:
        return None
    theta = {}
    for a, b in zip(expr1[1:], expr2[1:]):
        a, b = dereference(a, theta), dereference(b, theta)
        if a == b:
            continue
        if is_variable(a):
            theta[a] = b
        elif is_variable(b):
            theta[b] = a
        else:
            return None
    # Normalize chained substitutions.
    return {var: dereference(value, theta) for var, value in theta.items()}

theta = {"x": "John", "y": "Mary"}
expr = ("Loves", "x", "y")
print("Substitution example:", apply_substitution(expr, theta))

tests = [
    (("P", "x", "y"), ("P", "a", "b"), "Prediction: {x: a, y: b}"),
    (("Q", "x", "y"), ("Q", "Ali", "KRR"), "Prediction: {x: Ali, y: KRR}"),
    (("R", "x", "x"), ("R", "a", "b"), "Prediction: failure, because x would need two values"),
]

print("\nExercise 4.1:")
for left, right, prediction in tests:
    print(prediction)
    print(left, "and", right, "=>", unify(left, right))

Substitution example: ('Loves', 'John', 'Mary')

Exercise 4.1:
Prediction: {x: a, y: b}
('P', 'x', 'y') and ('P', 'a', 'b') => {'x': 'a', 'y': 'b'}
Prediction: {x: Ali, y: KRR}
('Q', 'x', 'y') and ('Q', 'Ali', 'KRR') => {'x': 'Ali', 'y': 'KRR'}
Prediction: failure, because x would need two values
('R', 'x', 'x') and ('R', 'a', 'b') => None


**Exercise 4.1 explanation.** The first two pairs have the MGUs ({x/a,y/b}) and ({x/Ali,y/KRR}). The third pair fails because the repeated variable (x) must be replaced consistently, but it would have to equal both distinct constants (a) and (b).

**Challenge 4.2 - Occurs check.** Unifying (x) with (f(x)) would create the infinite term (x=f(f(f(\dots)))). A full unifier therefore performs an occurs check and rejects a substitution when the variable occurs inside the term assigned to it. The lab's tuple representation does not model nested function terms, so this is explained rather than implemented.

## Task 5 - Generalized Modus Ponens (GMP)

In [5]:
# Demonstration: Human(x) -> Mortal(x), Human(Socrates)
fact = ("Human", "Socrates")
rule_premise = ("Human", "x")
rule_conclusion = ("Mortal", "x")
theta = unify(rule_premise, fact)
print("Demonstration substitution:", theta)
print("Derived:", apply_substitution(rule_conclusion, theta))

# Exercise 5.1
facts = [("Student", "Ali"), ("Studies", "Ali", "KRR")]
premise1 = ("Student", "x")
premise2 = ("Studies", "x", "KRR")
conclusion = ("EngagedLearner", "x")

theta1 = unify(premise1, facts[0])
instantiated_premise2 = apply_substitution(premise2, theta1)
theta2 = unify(instantiated_premise2, facts[1])
combined_theta = dict(theta1)
if theta2:
    combined_theta.update(theta2)

print("\nExercise 5.1:")
print("Match Student(x) with Student(Ali):", theta1)
print("After substitution, second premise is:", instantiated_premise2)
print("Match second premise with Studies(Ali, KRR):", theta2)
print("Combined substitution:", combined_theta)
print("Derived:", apply_substitution(conclusion, combined_theta))

Demonstration substitution: {'x': 'Socrates'}
Derived: ('Mortal', 'Socrates')

Exercise 5.1:
Match Student(x) with Student(Ali): {'x': 'Ali'}
After substitution, second premise is: ('Studies', 'Ali', 'KRR')
Match second premise with Studies(Ali, KRR): {}
Combined substitution: {'x': 'Ali'}
Derived: ('EngagedLearner', 'Ali')


**Logical meaning.** The common substitution (x=Ali) satisfies both premises, so Generalized Modus Ponens derives `EngagedLearner(Ali)`.

## Task 6 - Forward Chaining

In [6]:
def forward_chain(facts, rules):
    facts = set(facts)
    changed = True
    while changed:
        changed = False
        for premises, conclusion in rules:
            if premises.issubset(facts) and conclusion not in facts:
                print("Firing rule:", premises, "->", conclusion)
                facts.add(conclusion)
                changed = True
    return facts

print("Manual example:")
example_derived = forward_chain(
    {"P", "Q"},
    [({"P", "Q"}, "R"), ({"R"}, "S"), ({"S", "P"}, "T")],
)
print("Final facts:", example_derived)
print("Is T derived?", "T" in example_derived)

print("\nExercise 6.1:")
exercise_rules = [({"A"}, "C"), ({"B", "C"}, "D"), ({"D"}, "E")]
exercise_derived = forward_chain({"A", "B"}, exercise_rules)
print("Final facts:", exercise_derived)
print("Goal E derived?", "E" in exercise_derived)

Manual example:
Firing rule: {'Q', 'P'} -> R
Firing rule: {'R'} -> S
Firing rule: {'S', 'P'} -> T
Final facts: {'Q', 'S', 'P', 'R', 'T'}
Is T derived? True

Exercise 6.1:
Firing rule: {'A'} -> C
Firing rule: {'C', 'B'} -> D
Firing rule: {'D'} -> E
Final facts: {'C', 'A', 'D', 'B', 'E'}
Goal E derived? True


**Logical meaning.** Starting with (A) and (B), the rules fire in the required order: first (C), then (D), then (E). Thus the knowledge base entails the goal (E).

## Task 7 - Backward Chaining

In [7]:
def backward_chain(goal, facts, rules, visited=None, depth=0):
    if visited is None:
        visited = set()
    indent = "  " * depth
    if goal in facts:
        print(f"{indent}{goal} is a known fact")
        return True
    if goal in visited:
        print(f"{indent}Cycle detected while proving {goal}")
        return False
    visited.add(goal)
    matching_rules = [(p, c) for p, c in rules if c == goal]
    if not matching_rules:
        print(f"{indent}No fact or rule proves {goal}")
    for premises, _ in matching_rules:
        print(f"{indent}To prove {goal}, need {sorted(premises)}")
        if all(backward_chain(p, facts, rules, visited.copy(), depth + 1)
               for p in sorted(premises)):
            print(f"{indent}Therefore {goal} is proved")
            return True
    return False

example_rules = [({"P", "Q"}, "R"), ({"R"}, "S"), ({"S", "P"}, "T")]
print("Manual example:")
print("Can we prove T?", backward_chain("T", {"P", "Q"}, example_rules))

print("\nExercise 7.1:")
print("Can we prove E?", backward_chain("E", {"A", "B"}, exercise_rules))

Manual example:
To prove T, need ['P', 'S']
  P is a known fact
  To prove S, need ['R']
    To prove R, need ['P', 'Q']
      P is a known fact
      Q is a known fact
    Therefore R is proved
  Therefore S is proved
Therefore T is proved
Can we prove T? True

Exercise 7.1:
To prove E, need ['D']
  To prove D, need ['B', 'C']
    B is a known fact
    To prove C, need ['A']
      A is a known fact
    Therefore C is proved
  Therefore D is proved
Therefore E is proved
Can we prove E? True


### Exercise 7.2 - Forward versus backward chaining

Forward chaining is data-driven: it starts with all known facts and fires every applicable rule until reaching a fixed point. It may derive facts irrelevant to a particular query because it explores all consequences available from the data. Backward chaining is goal-driven: it starts with a specific query and recursively investigates only rules that could prove it. Therefore, backward chaining is usually more focused when one specific goal is given, while forward chaining is useful when many conclusions or later queries are expected.

## Task 8 - Mini Automated Reasoning System

Facts and predicates are tuples. A rule is a pair containing a tuple of premises and one conclusion. The engine matches premises using substitutions and repeatedly adds instantiated conclusions until no new facts remain.

In [8]:
from itertools import product

def merge_substitutions(base, extra):
    merged = dict(base)
    for variable, value in extra.items():
        value = dereference(value, merged)
        if variable in merged and dereference(merged[variable], merged) != value:
            return None
        merged[variable] = value
    return merged

def match_premises(premises, facts, theta=None):
    # Yield substitutions that simultaneously satisfy every premise.
    theta = {} if theta is None else theta
    if not premises:
        yield theta
        return
    first = apply_substitution(premises[0], theta)
    for fact in sorted(facts):
        local = unify(first, fact)
        if local is not None:
            merged = merge_substitutions(theta, local)
            if merged is not None:
                yield from match_premises(premises[1:], facts, merged)

def fol_forward_chain(initial_facts, rules):
    facts = set(initial_facts)
    trace = []
    changed = True
    while changed:
        changed = False
        for premises, conclusion in rules:
            for theta in list(match_premises(premises, facts)):
                derived = apply_substitution(conclusion, theta)
                if derived not in facts and not any(is_variable(x) for x in derived):
                    facts.add(derived)
                    trace.append((premises, theta, derived))
                    print("Rule:", premises, "->", conclusion)
                    print("  substitution:", theta)
                    print("  newly derived:", derived)
                    changed = True
    return facts, trace

initial_facts = {
    ("Student", "Ali"),
    ("Studies", "Ali", "KRR"),
    ("Passed", "Ali", "KRR"),
    ("Student", "Sara"),
    ("Studies", "Sara", "AI"),
}

university_rules = [
    ((("Passed", "x", "KRR"),), ("EligibleForCredits", "x")),
    ((("Student", "x"), ("Studies", "x", "KRR")), ("KRRStudent", "x")),
    ((("KRRStudent", "x"), ("EligibleForCredits", "x")), ("ReadyForCompletion", "x")),
]

print("Forward-chaining trace:")
derived_facts, trace = fol_forward_chain(initial_facts, university_rules)

queries = [
    ("EligibleForCredits", "Ali"),
    ("KRRStudent", "Ali"),
    ("ReadyForCompletion", "Ali"),
    ("ReadyForCompletion", "Sara"),
]

print("\nQuery answers:")
for query in queries:
    print(query, "=>", query in derived_facts)

print("\nWhy Sara's query is unproved:")
print("Sara studies AI, but the rule requires Studies(Sara, KRR), and there is no fact")
print("Passed(Sara, KRR). Therefore neither KRRStudent(Sara) nor")
print("EligibleForCredits(Sara) can be derived, so ReadyForCompletion(Sara) remains unproved.")

Forward-chaining trace:
Rule: (('Passed', 'x', 'KRR'),) -> ('EligibleForCredits', 'x')
  substitution: {'x': 'Ali'}
  newly derived: ('EligibleForCredits', 'Ali')
Rule: (('Student', 'x'), ('Studies', 'x', 'KRR')) -> ('KRRStudent', 'x')
  substitution: {'x': 'Ali'}
  newly derived: ('KRRStudent', 'Ali')
Rule: (('KRRStudent', 'x'), ('EligibleForCredits', 'x')) -> ('ReadyForCompletion', 'x')
  substitution: {'x': 'Ali'}
  newly derived: ('ReadyForCompletion', 'Ali')

Query answers:
('EligibleForCredits', 'Ali') => True
('KRRStudent', 'Ali') => True
('ReadyForCompletion', 'Ali') => True
('ReadyForCompletion', 'Sara') => False

Why Sara's query is unproved:
Sara studies AI, but the rule requires Studies(Sara, KRR), and there is no fact
Passed(Sara, KRR). Therefore neither KRRStudent(Sara) nor
EligibleForCredits(Sara) can be derived, so ReadyForCompletion(Sara) remains unproved.


**Logical meaning.** For Ali, forward chaining derives `EligibleForCredits`, `KRRStudent`, and finally `ReadyForCompletion`. Sara's existing facts match neither KRR-specific premise, so the final query is not entailed under the closed set of supplied rules and facts.

### Optional extension

Adding the two facts `Studies(Sara, KRR)` and `Passed(Sara, KRR)` is sufficient to satisfy both prerequisite branches and make `ReadyForCompletion(Sara)` derivable. (One fact alone would not establish both `KRRStudent(Sara)` and `EligibleForCredits(Sara)`.)

In [9]:
extended_facts = initial_facts | {
    ("Studies", "Sara", "KRR"),
    ("Passed", "Sara", "KRR"),
}
print("Optional extension trace:")
extended_result, extended_trace = fol_forward_chain(extended_facts, university_rules)
print("ReadyForCompletion(Sara) =>", ("ReadyForCompletion", "Sara") in extended_result)

Optional extension trace:
Rule: (('Passed', 'x', 'KRR'),) -> ('EligibleForCredits', 'x')
  substitution: {'x': 'Ali'}
  newly derived: ('EligibleForCredits', 'Ali')
Rule: (('Passed', 'x', 'KRR'),) -> ('EligibleForCredits', 'x')
  substitution: {'x': 'Sara'}
  newly derived: ('EligibleForCredits', 'Sara')
Rule: (('Student', 'x'), ('Studies', 'x', 'KRR')) -> ('KRRStudent', 'x')
  substitution: {'x': 'Ali'}
  newly derived: ('KRRStudent', 'Ali')
Rule: (('Student', 'x'), ('Studies', 'x', 'KRR')) -> ('KRRStudent', 'x')
  substitution: {'x': 'Sara'}
  newly derived: ('KRRStudent', 'Sara')
Rule: (('KRRStudent', 'x'), ('EligibleForCredits', 'x')) -> ('ReadyForCompletion', 'x')
  substitution: {'x': 'Ali'}
  newly derived: ('ReadyForCompletion', 'Ali')
Rule: (('KRRStudent', 'x'), ('EligibleForCredits', 'x')) -> ('ReadyForCompletion', 'x')
  substitution: {'x': 'Sara'}
  newly derived: ('ReadyForCompletion', 'Sara')
ReadyForCompletion(Sara) => True


## Post-lab summary

- Rules of inference derive conclusions whose premises are known to hold.
- CNF expresses a formula as a conjunction of disjunctive clauses, enabling resolution.
- Resolution proves a goal by adding its negation and deriving the empty clause.
- Substitution replaces variables; unification finds a consistent most general substitution.
- Generalized Modus Ponens extends implication-based inference to first-order predicates.
- Forward chaining is data-driven, while backward chaining is goal-driven.
- The integrated system combines matching, substitution, and Horn-rule forward chaining to answer queries.

The supplied manual mentions “Post-Lab Questions” in the deliverables but does not provide a separate post-lab question list. All questions and comparison prompts that do appear in Tasks 1-8 are answered above.